In [1]:
pip install pandas plotly nbformat nbclient ipywidgets matplotlib "urllib3<2"

You should consider upgrading via the '/Users/stripura/Desktop/ocp-workload-analysis-/venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import json
import os
import plotly.express as px
import plotly.graph_objects as go

# --- 1. CONFIGURATION & ENVIRONMENT SETUP ---
# Fetch paths from environment variables or default to current directory
config_dir = os.getenv('CONFIG_DIR', './configs')
data_dir = os.getenv('DATA_DIR', './sample_data')

# Construct full file paths
workload_data_path = os.path.join(data_dir, 'k8s_workload_inventory.csv')
complexity_cfg_path = os.path.join(config_dir, 'workload_complexity_config.json')
migration_cfg_path = os.path.join(config_dir, 'migration_config.json')

print(f"📂 Loading data from: {data_dir}")
print(f"⚙️ Loading configs from: {config_dir}")

# --- 2. LOAD DATA & CONFIGS ---
try:
    df = pd.read_csv(workload_data_path)
    
    with open(complexity_cfg_path) as f:
        comp_cfg = json.load(f)
        
    with open(migration_cfg_path) as f:
        mig_cfg = json.load(f)
except FileNotFoundError as e:
    print(f"❌ Critical Error: Could not find file. {e}")
    raise

# Data Cleaning: Handle missing values and convert Milli/MiB to standard numbers
df['CPU_Request_Milli'] = pd.to_numeric(df['CPU_Request_Milli'], errors='coerce').replace(-1, 0).fillna(0)
df['Mem_Request_MiB'] = pd.to_numeric(df['Mem_Request_MiB'], errors='coerce').replace(-1, 0).fillna(0)

# --- 3. CALCULATE COMPLEXITY SCORE ---
def calculate_workload_score(row, cfg):
    w = cfg['weights']
    res = cfg['resource_intensity']
    score = 0
    
    # Infrastructure Gravity logic
    if str(row['Storage_Complexity']).upper() == 'RWX': 
        score += w['infrastructure_gravity']['storage_rwx_shared']
    if str(row['Host_Network']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['host_network_enabled']
    if str(row['Privileged']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['privileged_mode']
    if row['Additional_Networks'] > 0: 
        score += w['infrastructure_gravity']['additional_networks_multus']
    
    # Resource Intensity (CPU/Memory Buckets)
    if row['CPU_Request_Milli'] >= res['cpu']['high_threshold_milli']: 
        score += res['cpu']['points']['high']
    if row['Mem_Request_MiB'] >= res['memory']['high_threshold_mib']: 
        score += res['memory']['points']['high']
    
    # Reliability Risk (Missing Probes)
    if str(row['Liveness_Probe']).capitalize() == 'No': 
        score += w['reliability_risk']['missing_liveness_probe']
    if str(row['Readiness_Probe']).capitalize() == 'No': 
        score += w['reliability_risk']['missing_readiness_probe']
    
    return min(score, 100)

df['Complexity_Score'] = df.apply(lambda r: calculate_workload_score(r, comp_cfg), axis=1)

# --- 4. ASSIGN STRATEGY (GITOPS AWARE) ---
def assign_strategy(row, m_cfg):
    score = row['Complexity_Score']
    
    # GitOps Readiness Pre-check
    is_gitops_ready = (
        str(row['Liveness_Probe']).capitalize() == 'Yes' and 
        str(row['Host_Network']).capitalize() == 'No' and 
        str(row['Privileged']).capitalize() == 'No'
    )
    
    # Strategy mapping from config thresholds
    strat = m_cfg['migration_strategies']
    
    if is_gitops_ready and score <= strat['GITOPS_REHOST']['max_complexity_score']:
        return "GITOPS_REHOST"
    elif score <= strat['REHOST']['max_complexity_score']:
        return "REHOST"
    elif score <= strat['REPLATFORM']['max_complexity_score']:
        return "REPLATFORM"
    else:
        return "REARCHITECT"

df['Suggested_Strategy'] = df.apply(lambda r: assign_strategy(r, mig_cfg), axis=1)

print("✅ Scoring and Strategy assignment complete.")

📂 Loading data from: ./sample_data
⚙️ Loading configs from: ./configs
❌ Critical Error: Could not find file. [Errno 2] No such file or directory: './sample_data/k8s_workload_inventory.csv'


FileNotFoundError: [Errno 2] No such file or directory: './sample_data/k8s_workload_inventory.csv'

In [3]:
import pandas as pd
import numpy as np
import json
import os
import plotly.express as px
import plotly.graph_objects as go

# --- 1. CONFIGURATION & PATH SETUP ---
# Explicitly pointing to your directory structure
config_dir = './configs'
data_dir = './sample_data'

# File paths
workload_data_path = os.path.join(data_dir, 'k8s_workload_inventory.csv')
complexity_cfg_path = os.path.join(config_dir, 'workload_complexity_config.json')
migration_cfg_path = os.path.join(config_dir, 'migration_config.json')

# --- 2. LOAD DATA & CONFIGS ---
try:
    df = pd.read_csv(workload_data_path)
    
    with open(complexity_cfg_path) as f:
        comp_cfg = json.load(f)
        
    with open(migration_cfg_path) as f:
        mig_cfg = json.load(f)
        
    print(f"✅ Successfully loaded {len(df)} workloads from {workload_data_path}")
except FileNotFoundError as e:
    print(f"❌ Error: Check if files exist in {data_dir} or {config_dir}")
    raise

# Data Cleaning: Handle -1 (undefined) and fill missing probes/security fields
df['CPU_Request_Milli'] = df['CPU_Request_Milli'].replace(-1, 0).fillna(0)
df['Mem_Request_MiB'] = df['Mem_Request_MiB'].replace(-1, 0).fillna(0)
for col in ['Liveness_Probe', 'Readiness_Probe', 'Host_Network', 'Privileged']:
    df[col] = df[col].fillna('No')

# --- 3. SCORING ENGINE ---
def calculate_workload_score(row, cfg):
    w = cfg['weights']
    res = cfg['resource_intensity']
    score = 0
    
    # Storage & Infrastructure Gravity
    if str(row['Storage_Complexity']).upper() == 'RWX': 
        score += w['infrastructure_gravity']['storage_rwx_shared']
    if str(row['Host_Network']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['host_network_enabled']
    if str(row['Privileged']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['privileged_mode']
    
    # Resource Intensity (CPU/Memory High-Thresholds)
    if row['CPU_Request_Milli'] >= res['cpu']['high_threshold_milli']: 
        score += res['cpu']['points']['high']
    if row['Mem_Request_MiB'] >= res['memory']['high_threshold_mib']: 
        score += res['memory']['points']['high']
    
    # Reliability Risk
    if str(row['Liveness_Probe']).capitalize() == 'No': 
        score += w['reliability_risk']['missing_liveness_probe']
    
    return min(score, 100)

df['Complexity_Score'] = df.apply(lambda r: calculate_workload_score(r, comp_cfg), axis=1)

# --- 4. STRATEGY ASSIGNMENT (GITOPS AWARE) ---
def assign_strategy(row, m_cfg):
    score = row['Complexity_Score']
    strat_cfg = m_cfg['migration_strategies']
    
    # Check GitOps Readiness
    is_gitops_ready = (
        str(row['Liveness_Probe']).capitalize() == 'Yes' and 
        str(row['Host_Network']).capitalize() == 'No' and 
        str(row['Privileged']).capitalize() == 'No'
    )
    
    if is_gitops_ready and score <= strat_cfg['GITOPS_REHOST']['max_complexity_score']:
        return "GITOPS_REHOST"
    elif score <= strat_cfg['REHOST']['max_complexity_score']:
        return "REHOST"
    elif score <= strat_cfg['REPLATFORM']['max_complexity_score']:
        return "REPLATFORM"
    else:
        return "REARCHITECT"

df['Suggested_Strategy'] = df.apply(lambda r: assign_strategy(r, mig_cfg), axis=1)

# --- 5. VISUALIZATIONS ---

# Pie Chart: Strategy Distribution
fig_pie = px.pie(df, names='Suggested_Strategy', 
                 title="Migration Strategy Distribution",
                 color_discrete_sequence=px.colors.qualitative.Pastel)
fig_pie.show()

# Bar Chart: Workload Counts by Kind
fig_bar = px.bar(df['Kind'].value_counts().reset_index(), 
                 x='Kind', y='count', color='Kind',
                 title="Workload Counts by Kind",
                 labels={'count': 'Total Count', 'Kind': 'Workload Type'})
fig_bar.show()

# --- 6. TOP 25 LISTS ---
print("\n🟢 TOP 25 EASY WORKLOADS (Sorted by Score)")
display(df.sort_values(by=['Complexity_Score', 'Name']).head(25)[['Name', 'Namespace', 'Complexity_Score', 'Suggested_Strategy']])

print("\n🔴 TOP 25 DIFFICULT WORKLOADS (Sorted by Score)")
display(df.sort_values(by=['Complexity_Score', 'Name'], ascending=[False, True]).head(25)[['Name', 'Namespace', 'Complexity_Score', 'Suggested_Strategy']])

❌ Error: Check if files exist in ./sample_data or ./configs


FileNotFoundError: [Errno 2] No such file or directory: './sample_data/k8s_workload_inventory.csv'

In [7]:
import os
import pandas as pd
import json
from pathlib import Path

# --- 1. CLEAN PATH RESOLVER ---
# Get the absolute path of the directory containing this notebook
base_dir = Path(os.getcwd()).resolve()

# Define the two SEPARATE top-level directories
# Based on your 'ls', these are both in the same root folder
data_dir = base_dir / "sample_data"
config_dir = base_dir / "configs"

# Define the individual files
workload_data_path = data_dir / "k8s_workload_inventory.csv"
complexity_cfg_path = config_dir / "workload_complexity_config.json"
migration_cfg_path = config_dir / "migration_config.json"

print(f"📍 Project Root: {base_dir}")
print(f"📂 Looking for Data in: {data_dir}")
print(f"⚙️ Looking for Configs in: {config_dir}")

# --- 2. VERIFY AND LOAD ---
try:
    # Check Data File
    if not workload_data_path.exists():
        raise FileNotFoundError(f"Missing CSV at: {workload_data_path}")
    
    # Check Config Files
    if not complexity_cfg_path.exists():
        raise FileNotFoundError(f"Missing JSON at: {complexity_cfg_path}")
    
    if not migration_cfg_path.exists():
        raise FileNotFoundError(f"Missing JSON at: {migration_cfg_path}")

    # Load the files
    df = pd.read_csv(workload_data_path)
    
    with open(complexity_cfg_path, 'r') as f:
        comp_cfg = json.load(f)
        
    with open(migration_cfg_path, 'r') as f:
        mig_cfg = json.load(f)

    print(f"\n✅ SUCCESS: Loaded {len(df)} workloads and all configurations.")

except FileNotFoundError as e:
    print(f"\n❌ PATH ERROR: {e}")
    # Optional: list directory to help you see where you actually are
    print(f"🔍 Items in Root: {os.listdir(base_dir)}")
    raise

# Data Cleaning: Handle -1 (undefined) and fill missing probes/security fields
df['CPU_Request_Milli'] = df['CPU_Request_Milli'].replace(-1, 0).fillna(0)
df['Mem_Request_MiB'] = df['Mem_Request_MiB'].replace(-1, 0).fillna(0)
for col in ['Liveness_Probe', 'Readiness_Probe', 'Host_Network', 'Privileged']:
    df[col] = df[col].fillna('No')

# --- 3. SCORING ENGINE ---
def calculate_workload_score(row, cfg):
    w = cfg['weights']
    res = cfg['resource_intensity']
    score = 0
    
    # Storage & Infrastructure Gravity
    if str(row['Storage_Complexity']).upper() == 'RWX': 
        score += w['infrastructure_gravity']['storage_rwx_shared']
    if str(row['Host_Network']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['host_network_enabled']
    if str(row['Privileged']).capitalize() == 'Yes': 
        score += w['infrastructure_gravity']['privileged_mode']
    
    # Resource Intensity (CPU/Memory High-Thresholds)
    if row['CPU_Request_Milli'] >= res['cpu']['high_threshold_milli']: 
        score += res['cpu']['points']['high']
    if row['Mem_Request_MiB'] >= res['memory']['high_threshold_mib']: 
        score += res['memory']['points']['high']
    
    # Reliability Risk
    if str(row['Liveness_Probe']).capitalize() == 'No': 
        score += w['reliability_risk']['missing_liveness_probe']
    
    return min(score, 100)

df['Complexity_Score'] = df.apply(lambda r: calculate_workload_score(r, comp_cfg), axis=1)

# --- 4. STRATEGY ASSIGNMENT (GITOPS AWARE) ---
def assign_strategy(row, m_cfg):
    score = row['Complexity_Score']
    strat_cfg = m_cfg['migration_strategies']
    
    # Check GitOps Readiness
    is_gitops_ready = (
        str(row['Liveness_Probe']).capitalize() == 'Yes' and 
        str(row['Host_Network']).capitalize() == 'No' and 
        str(row['Privileged']).capitalize() == 'No'
    )
    
    if is_gitops_ready and score <= strat_cfg['GITOPS_REHOST']['max_complexity_score']:
        return "GITOPS_REHOST"
    elif score <= strat_cfg['REHOST']['max_complexity_score']:
        return "REHOST"
    elif score <= strat_cfg['REPLATFORM']['max_complexity_score']:
        return "REPLATFORM"
    else:
        return "REARCHITECT"

df['Suggested_Strategy'] = df.apply(lambda r: assign_strategy(r, mig_cfg), axis=1)

# --- 5. VISUALIZATIONS ---

# Pie Chart: Strategy Distribution
fig_pie = px.pie(df, names='Suggested_Strategy', 
                 title="Migration Strategy Distribution",
                 color_discrete_sequence=px.colors.qualitative.Pastel)
fig_pie.show()

# Bar Chart: Workload Counts by Kind
fig_bar = px.bar(df['Kind'].value_counts().reset_index(), 
                 x='Kind', y='count', color='Kind',
                 title="Workload Counts by Kind",
                 labels={'count': 'Total Count', 'Kind': 'Workload Type'})
fig_bar.show()

# --- 6. TOP 25 LISTS ---
print("\n🟢 TOP 25 EASY WORKLOADS (Sorted by Score)")
display(df.sort_values(by=['Complexity_Score', 'Name']).head(25)[['Name', 'Namespace', 'Complexity_Score', 'Suggested_Strategy']])

print("\n🔴 TOP 25 DIFFICULT WORKLOADS (Sorted by Score)")
display(df.sort_values(by=['Complexity_Score', 'Name'], ascending=[False, True]).head(25)[['Name', 'Namespace', 'Complexity_Score', 'Suggested_Strategy']])

📍 Project Root: /Users/stripura/Desktop/ocp-workload-analysis-/configs
📂 Looking for Data in: /Users/stripura/Desktop/ocp-workload-analysis-/configs/sample_data
⚙️ Looking for Configs in: /Users/stripura/Desktop/ocp-workload-analysis-/configs/configs

❌ PATH ERROR: Missing CSV at: /Users/stripura/Desktop/ocp-workload-analysis-/configs/sample_data/k8s_workload_inventory.csv
🔍 Items in Root: ['workload_complexity_config.json', 'migration_config.json', 'migration1.ipynb', 'ns_complexity_config.json', '.ipynb_checkpoints']


FileNotFoundError: Missing CSV at: /Users/stripura/Desktop/ocp-workload-analysis-/configs/sample_data/k8s_workload_inventory.csv

In [8]:
import os
import pandas as pd
import json
from pathlib import Path

# --- 1. BULLETPROOF PATH RESOLVER ---
# This finds where the notebook IS, not where the user STARTED it.
try:
    # If running in a standard Jupyter environment
    base_dir = Path(os.path.abspath('')).resolve()
except:
    base_dir = Path(os.getcwd()).resolve()

# Check for 'sample_data' at current level. If not there, go up one level.
# This fixes cases where Jupyter starts inside a subfolder.
if not (base_dir / "sample_data").exists() and (base_dir.parent / "sample_data").exists():
    base_dir = base_dir.parent

data_dir = base_dir / "sample_data"
config_dir = base_dir / "configs"

# Individual File Paths
workload_data_path = data_dir / "k8s_workload_inventory.csv"
complexity_cfg_path = config_dir / "workload_complexity_config.json"
migration_cfg_path = config_dir / "migration_config.json"

print(f"📍 Base Directory Found: {base_dir}")

# ERROR CHECKING BLOCK
missing_files = []
for p in [workload_data_path, complexity_cfg_path, migration_cfg_path]:
    if not p.exists():
        missing_files.append(str(p))

if missing_files:
    print("\n❌ STOP! The following files were not found:")
    for f in missing_files:
        print(f"  - {f}")
    
    print(f"\n🔍 DEBUG INFO:")
    print(f"  - Currently in: {os.getcwd()}")
    if base_dir.exists():
        print(f"  - Items in {base_dir.name}: {os.listdir(base_dir)}")
    raise FileNotFoundError("Check the file paths printed above.")

📍 Base Directory Found: /Users/stripura/Desktop/ocp-workload-analysis-
